# Reflective surface testing

This notebook tests the backend setup for a diffuse surface without an
atmosphere.

In [ ]:
import eradiate
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import xarray as xr
from eradiate.experiments import AtmosphereExperiment
from eradiate.units import unit_registry as ureg

import eradiate_disort as ed

sns.set_theme(style="ticks")

# eradiate.set_mode("mono")
eradiate.set_mode("ckd")

SPP = 100_000
if eradiate.get_mode().is_ckd:
    SPP //= 16

In [ ]:
def get_experiment(sza: float = 0.0):
    return AtmosphereExperiment(
        geometry={
            "type": "plane_parallel",
            "zgrid": (np.arange(0, 120.001, 1.0) * ureg.km).to("m"),
        },
        surface={"type": "lambertian", "reflectance": 0.5},
        atmosphere=None,
        illumination={"type": "directional", "zenith": sza, "azimuth": 0.0},
        measures={
            "type": "mdistant",
            "construct": "hplane",
            "azimuth": 0.0,
            "zeniths": np.arange(-75.0, 76.0, 1.0),
            "srf": {"type": "delta", "wavelengths": [550.0]},
        },
    )


def reindex_pplane(da: xr.DataArray):
    result = da.stack(i=("x_index", "y_index")).drop_vars(("i", "x_index", "y_index"))
    mask_negative = result["vaa"] == 180.0
    mask_nadir = result["vza"] == 0.0
    neg = result.where(mask_negative & ~mask_nadir).dropna("i")
    neg["vza"] *= -1.0
    pos = result.where(~mask_negative).dropna("i")
    result = xr.concat((neg, pos), dim="i").sortby("vza")
    result = result.squeeze()
    return result

In [ ]:
exp = get_experiment(30.0)
result_mitsuba = eradiate.run(exp, spp=SPP)["radiance"].squeeze()

backend = ed.EradiateDisortBackend()
result_disort = reindex_pplane(backend.run(exp))
# result_disort

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 3), layout="constrained")

ax.plot(result_mitsuba["vza"], result_mitsuba, label="Mitsuba")
ax.plot(result_disort["vza"], result_disort, label="CDISORT", ls="--")
ax.set_xlabel("θ [°]")
ax.set_ylabel("Radiance [W/m²/sr]")
ax.set_ylim([-0.05, 0.65])
ax.legend()

plt.show()